# Download dei dati dal mercato italiano

In [ ]:
## SETUP: librerie e percorsi

import os
import time
import warnings
import numpy as np
import pandas as pd
import yfinance as yf

from tqdm import tqdm

# Setto i percorsi alle cartelle
BASE_DIR = ".."

DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MANUAL_DIR = os.path.join(DATA_DIR, "manual")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

START_DATE = "2015-01-01" # periodo inizio
END_DATE = "2025-01-01"  # esclusivo, quindi include tutto il 2024

In [ ]:
# riporto i dati relativi ai ticker, ovvero codici identificativi di aziende nella libreria yahoo finance

tickers_data = [
    ["A2A", "A2A.MI", "A2A", "Utilities"],
    ["AMP", "AMP.MI", "Amplifon", "Healthcare"],
    ["AZM", "AZM.MI", "Azimut", "Financial Services"],
    ["BAMI", "BAMI.MI", "Banco BPM", "Banking"],
    ["BMED", "BMED.MI", "Banca Mediolanum", "Financial Services"],
    ["BMPS", "BMPS.MI", "Banca Monte dei Paschi di Siena", "Banking"],
    ["BPE", "BPE.MI", "BPER Banca", "Banking"],
    ["BC", "BC.MI", "Brunello Cucinelli", "Consumer Goods"],
    ["BZU", "BZU.MI", "Buzzi", "Construction Materials"],
    ["CPR", "CPR.MI", "Campari", "Food Beverage"],
    ["DIA", "DIA.MI", "Diasorin", "Healthcare"],
    ["ENEL", "ENEL.MI", "Enel", "Utilities"],
    ["ENI", "ENI.MI", "ENI", "Energy"],
    ["ERG", "ERG.MI", "ERG", "Energy"],
    ["FBK", "FBK.MI", "FinecoBank", "Banking"],
    ["G", "G.MI", "Assicurazioni Generali", "Insurance"],
    ["HER", "HER.MI", "Hera", "Utilities"],
    ["IG", "IG.MI", "Italgas", "Utilities"],
    ["INW", "INW.MI", "Inwit", "Telecommunications"],
    ["IP", "IP.MI", "Interpump", "Industrials"],
    ["ISP", "ISP.MI", "Intesa Sanpaolo", "Banking"],
    ["LDO", "LDO.MI", "Leonardo", "Industrials"],
    ["MB", "MB.MI", "Mediobanca", "Banking"],
    ["MONC", "MONC.MI", "Moncler", "Consumer Goods"],
    ["NEXI", "NEXI.MI", "Nexi", "Financial Technology"],
    ["PIRC", "PIRC.MI", "Pirelli", "Consumer Goods"],
    ["PST", "PST.MI", "Poste Italiane", "Financial Services"],
    ["PRY", "PRY.MI", "Prysmian", "Industrials"],
    ["RACE", "RACE.MI", "Ferrari", "Automobiles"],
    ["REC", "REC.MI", "Recordati", "Healthcare"],
    ["SPM", "SPM.MI", "Saipem", "Energy"],
    ["SRG", "SRG.MI", "Snam", "Utilities"],
    ["STLAM", "STLAM.MI", "Stellantis", "Automobiles"],
    ["STM", "STMMI.MI", "STMicroelectronics", "Technology"],
    ["TIT", "TIT.MI", "Telecom Italia", "Telecommunications"],
    ["TEN", "TEN.MI", "Tenaris", "Energy"],
    ["TRN", "TRN.MI", "Terna", "Utilities"],
    ["UCG", "UCG.MI", "UniCredit", "Banking"],
    ["UNI", "UNI.MI", "Unipol", "Insurance"],
]

# imposto i nomi delle colonne del DataFrame dei ticker
tickers = pd.DataFrame(
    tickers_data,
    columns=["symbol", "yahoo_ticker", "name", "sector"]
)

tickers_path = os.path.join(MANUAL_DIR, "tickers_italia_moderni.csv") # percorso tickers
tickers.to_csv(tickers_path, index=False) # salvo il DataFrame in un csv senza indici

tickers # stampo tabella dei tickers

In [ ]:
def download_raw_market_data(yahoo_ticker, start=START_DATE, end=END_DATE): # yahoo_ticker è il ticker usato da Yahoo Finance
    """
    Scarica dati grezzi da Yahoo Finance:
    - Open, High, Low, Close (prezzo di apertura, prezzo max, prezzo min, prezzo di chiusura)
    - Volume (numero di azioni scambiate nella giornata)
    - Dividends (dividendo distribuito per azione, ovvero una porzione di utili distribuiti dalla società agli azionisti)
    - Stock Splits (eventuale frazionamento o raggruppamento azionario)

    Non usa Adjusted Close per le analisi (viene tenuto per confronto).
    """
    
    ticker = yf.Ticker(yahoo_ticker) # creo l'oggetto ticker relativo all'azienda
    
    # scarico i dati 
    df = ticker.history(
        start=start,
        end=end,
        auto_adjust=False, # necessario, altrimenti i prezzi verrebbero corretti automaticamente
        actions=True # per avere anche dividendi e stock splits
    )
    
    if df.empty:
        return pd.DataFrame()
    
    df = df.reset_index() # altrimenti non posso selezionare la data, poiché la data è l'indice stesso.
    
    if "Date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Date"}) # se la prima colonna non è Date, la rinomino io
        
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None) # trasformo i valori in date, rimuovendo i fusi orari
    
    required_cols = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close", # lo lascio anche se non lo usiamo, magari torna utile
        "Volume",
        "Dividends",
        "Stock Splits"
    ]
    
    # se la colonna non c'è nei dati scaricati, creo la colonna nel mio e ci metto tutti NaN
    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan
    
    df = df[required_cols].copy() # creo un nuovo dataframe con solo le colonne che mi interessano
    df["YahooTicker"] = yahoo_ticker # per ogni riga aggiungo il ticker yahoo di provenienza
    
    return df

In [ ]:
all_raw = []
failed_downloads = []

for _, row in tqdm(tickers.iterrows(), total=len(tickers)): # ciclo su tutti i ticker, riga per riga (indici non mi servono). In più, lo stampo a schermo
    symbol = row["symbol"]
    yahoo_ticker = row["yahoo_ticker"]
    
    try:
        df = download_raw_market_data(yahoo_ticker) # provo a scaricare i dati relativi al ticker
        
        if df.empty:
            failed_downloads.append((symbol, yahoo_ticker, "empty dataframe"))
            continue
            
        df["Symbol"] = symbol
        df["Name"] = row["name"]
        df["Sector"] = row["sector"]
        
        output_file = os.path.join(RAW_DIR, f"{symbol}_{yahoo_ticker}_raw.csv") # setto nome e path di output
        df.to_csv(output_file, index=False)
        
        all_raw.append(df) 
        
        time.sleep(0.3) # per evitare errori da sovraccarico di richieste
        
    except Exception as e:
        failed_downloads.append((symbol, yahoo_ticker, str(e)))

raw_prices = pd.concat(all_raw, ignore_index=True) # unisco i risultati

raw_prices_path = os.path.join(RAW_DIR, "italy_raw_prices_2015_2024.csv")
raw_prices.to_csv(raw_prices_path, index=False) # salvo nel file di output finale

failed_downloads_df = pd.DataFrame(
    failed_downloads,
    columns=["symbol", "yahoo_ticker", "error"]
)

failed_downloads_df.to_csv(
    os.path.join(RAW_DIR, "failed_downloads.csv"),
    index=False
)

print("Titoli scaricati:", raw_prices["Symbol"].nunique())
print("Download falliti:", len(failed_downloads_df))

failed_downloads_df # stampo eventuali download falliti

In [ ]:
def adjust_prices_like_paper(df_stock, symbol): # dataframe titolo, simbolo, eventuali correzioni
    """
    Corregge il prezzo Close grezzo seguendo la logica del paper (Non usa Adj Close):
    
    1. correzione per dividendi;
    2. correzione per stock split;
    3. correzione per fattori manuali aggiuntivi.
    """
    
    df = df_stock.copy() # nuovo dataframe su cui lavorare
    df = df.sort_values("Date").reset_index(drop=True) # ordiniamo in ordine cronologico
    
    df["P_raw"] = df["Close"].astype(float) # creo colonna del prezzo grezzo
    df["Volume"] = df["Volume"].astype(float) # creo colonna del volume
    
    # 1. Correzione per dividendi
    
    df["Dividends"] = df["Dividends"].fillna(0).astype(float) # rimpiazzo i Na con 0
    
    df["dividend_factor_daily"] = 1.0 # valore di default (nessuna correzione)
    
    mask_div = (
        (df["Dividends"] > 0) &
        (df["P_raw"] > 0) &
        (df["P_raw"].notna())
    ) # identifico le date con un booleano che mi indica la presenza di dividendi
    
    df.loc[mask_div, "dividend_factor_daily"] = (
        1 + df.loc[mask_div, "Dividends"] / df.loc[mask_div, "P_raw"]
    ) # se c'è il dividendo, correggo il prezzo come nel paper
    
    df["dividend_factor_cumulative"] = df["dividend_factor_daily"].cumprod() # calcolo la produttoria come nel paper
    
    df["P_after_dividends"] = (
        df["P_raw"] * df["dividend_factor_cumulative"]
    ) # salvo il valore finale
    
    # 2. Stock split da Yahoo
    
    df["Stock Splits"] = df["Stock Splits"].fillna(0).astype(float) 
    df["k_split"] = 1.0 # valore di default
    
    mask_split = df["Stock Splits"] > 0 # anche qui flaggo se c'è uno split 
    
    # Yahoo: 2.0 significa split 2:1.
    # Nel paper, se il prezzo si dimezza, k = 0.5.
    df.loc[mask_split, "k_split"] = 1 / df.loc[mask_split, "Stock Splits"]
    
    # 3. Applicazione correzioni    
    # Formula del paper: prodotto dei k_T per T > t.
    future_split_product_inclusive = df["k_split"].iloc[::-1].cumprod().iloc[::-1]
    df["split_factor_future"] = future_split_product_inclusive.shift(-1).fillna(1.0)
    
    df["P_adjusted_manual"] = (
        df["P_after_dividends"] * df["split_factor_future"]
    )
    
    return df

In [ ]:
def compute_log_returns_with_gap_rule(df_stock, max_gap=7): # dataframe del titolo + max gap fra due osservazioni
    """
    Calcola log-rendimenti usando il prezzo corretto manualmente.
    Se il prezzo precedente manca, cerca fino a max_gap osservazioni precedenti (come nel paper).
    """
    
    df = df_stock.copy()
    df = df.sort_values("Date").reset_index(drop=True)
    
    price = df["P_adjusted_manual"].astype(float) # estraggo il prezzo
    
    previous_valid_price = price.ffill(limit=max_gap).shift(1) # se nei max_gap giorni successivi c'è NaN, lo riempio con l'ultimo prezzo disponibile. poi shifto di uno "verso il basso", così ho effettivamente i prezzi precedenti
    
    df["log_return"] = np.log(price) - np.log(previous_valid_price) # calcolo il log-rendimento
    
    invalid = (
        price.isna() |
        previous_valid_price.isna() |
        (price <= 0) |
        (previous_valid_price <= 0)
    ) # creo una flag per evidenziare i casi non validi 
    
    df.loc[invalid, "log_return"] = np.nan # se il caso non è valido metto nan
    
    return df

In [ ]:
def compute_traded_money(df_stock):
    """
    Calcola il controvalore scambiato:
    traded_money = prezzo grezzo * volume.
    """
    
    df = df_stock.copy()
    df["traded_money"] = df["P_raw"] * df["Volume"] # per definizione di controvalore
    
    invalid = (
        df["P_raw"].isna() |
        df["Volume"].isna() |
        (df["P_raw"] <= 0) |
        (df["Volume"] < 0)
    ) # flaggo i casi non validi
    
    df.loc[invalid, "traded_money"] = np.nan 
    
    return df

In [ ]:
'''
    Ora, titolo per titolo, applichiamo le funzioni precedenti
'''

processed_list = []

for symbol in tqdm(raw_prices["Symbol"].unique()): # ciclo su tutti i simboli
    df_stock = raw_prices[raw_prices["Symbol"] == symbol].copy() # prendo da raw_prices solo le righe del titolo corrente
    
    df_stock = adjust_prices_like_paper(
        df_stock=df_stock,
        symbol=symbol
    )
    
    df_stock = compute_log_returns_with_gap_rule(df_stock, max_gap=7)
    df_stock = compute_traded_money(df_stock)
    
    processed_list.append(df_stock)

processed = pd.concat(processed_list, ignore_index=True) # concateno i risultati

processed_path = os.path.join(
    PROCESSED_DIR,
    "italy_prices_returns_traded_money_2015_2024.csv"
) # percorso output

processed.to_csv(processed_path, index=False)

processed.head() # stampo le prime 5 righe

In [ ]:
# CONTROLLO QUALITA

quality = (
    processed
    .groupby(["Symbol", "YahooTicker", "Name", "Sector"]) # raggruppo per titolo
    .agg(
        first_date=("Date", "min"), # data minima titolo
        last_date=("Date", "max"), #data max titolo
        n_prices=("P_raw", "count"), # numero di prezzi grezzi presenti
        n_returns=("log_return", "count"), # numero di rendimenti logaritmici
        missing_returns=("log_return", lambda x: x.isna().mean()), # ritorni mancanti
        n_dividends=("Dividends", lambda x: (x > 0).sum()),
        n_splits=("Stock Splits", lambda x: (x > 0).sum()),
        mean_volume=("Volume", "mean"), # volume di azioni scambiate
        mean_traded_money=("traded_money", "mean") # controvolare medio giornaliero scambiato
    )
    .reset_index() # riporto il group by a colonne normali
    .sort_values("n_returns", ascending=False) 
)

quality_path = os.path.join(
    PROCESSED_DIR,
    "data_quality_summary_italy_2015_2024.xlsx"
)

quality.to_excel(quality_path, index=False)

quality

In [ ]:
# Nel mio caso filtro per titoli che sono disponibili almeno per l'80% dell'intevallo osservato

total_trading_days = processed["Date"].nunique() # numero di date diverse nel dataset 

MIN_OBSERVATIONS = int(total_trading_days * 0.80) # soglia minima

eligible_symbols = (
    quality
    .query("n_returns >= @MIN_OBSERVATIONS") # seleziono solo i titoli che superano la soglia minima
    ["Symbol"]
    .tolist()
)

print("Giorni totali nel dataset:", total_trading_days)
print("Soglia minima osservazioni:", MIN_OBSERVATIONS)
print("Titoli eleggibili:", len(eligible_symbols))

eligible_symbols

In [ ]:
# creo la matrice dei log-ritorni

returns_matrix = (
    processed[processed["Symbol"].isin(eligible_symbols)] # prendiamo i titoli che superano la soglia minima
    .pivot(index="Date", columns="Symbol", values="log_return") # trasformo da formato lungo a formato matrice
    .sort_index() # ordiniamo la matrice in ordine cronologico
)

returns_matrix_path = os.path.join(
    PROCESSED_DIR,
    "returns_matrix_italy_2015_2024.csv"
)

returns_matrix.to_csv(returns_matrix_path)

missing_by_symbol = returns_matrix.isna().mean().sort_values(ascending=False) # percentuale di valori mancanti per ogni titolo

display(returns_matrix.head())
display(missing_by_symbol)

In [ ]:
# costruzione della matrice del controvalore scambiato

traded_money_matrix = (
    processed[processed["Symbol"].isin(eligible_symbols)] # seleziono titoli validi
    .pivot(index="Date", columns="Symbol", values="traded_money") # trasformo da formato lungo a formato matrice
    .sort_index()  # ordine cronologico
)

traded_money_matrix_path = os.path.join(
    PROCESSED_DIR,
    "traded_money_matrix_italy_2015_2024.csv"
)

traded_money_matrix.to_csv(traded_money_matrix_path)

traded_money_matrix.head()

In [ ]:
# COSTRUISCO BENCHMARK PER CAPITALIZZAZIONE

def download_shares_outstanding(yahoo_ticker, start=START_DATE, end=END_DATE):
    """
    Prova a scaricare il numero storico di azioni in circolazione da yfinance.
    """
    
    ticker = yf.Ticker(yahoo_ticker) # creo oggetto ticker
    
    try:
        shares = ticker.get_shares_full(start=start, end=end) # ottengo totale azioni in circolazione
        
        if shares is None or len(shares) == 0:
            return pd.DataFrame()
        
        shares = shares.reset_index() # trasformo data da indice a colonna normale
        shares.columns = ["Date", "shares_outstanding"] # di conseguenza, setto nomi colonne
        shares["Date"] = pd.to_datetime(shares["Date"]).dt.tz_localize(None) # converto in formato data pandas senza timezone
        
        return shares
    
    except Exception:
        return pd.DataFrame()

In [ ]:
shares_all = []
failed_shares = []

for _, row in tqdm(tickers.iterrows(), total=len(tickers)): # iteriamo sui tickers msotrando l'avanzamento
    symbol = row["symbol"]
    yahoo_ticker = row["yahoo_ticker"]
    
    shares = download_shares_outstanding(yahoo_ticker)
    
    if shares.empty:
        failed_shares.append((symbol, yahoo_ticker)) # se non è disponibile vado avanti
        continue
    
    shares["Symbol"] = symbol
    shares["YahooTicker"] = yahoo_ticker
    
    shares_all.append(shares)
    
    time.sleep(0.3) # per evitare problemi

if len(shares_all) > 0:
    shares_outstanding = pd.concat(shares_all, ignore_index=True) # concateno, se ci sono
else:
    shares_outstanding = pd.DataFrame(
        columns=["Date", "shares_outstanding", "Symbol", "YahooTicker"]
    )

shares_outstanding_path = os.path.join(
    RAW_DIR,
    "shares_outstanding_italy_2015_2024.csv"
)

shares_outstanding.to_csv(shares_outstanding_path, index=False)

failed_shares_df = pd.DataFrame(
    failed_shares,
    columns=["symbol", "yahoo_ticker"]
)

failed_shares_df.to_csv(
    os.path.join(RAW_DIR, "failed_shares_outstanding.csv"),
    index=False
)

print("Titoli con shares outstanding:", shares_outstanding["Symbol"].nunique())
print("Titoli senza shares outstanding:", len(failed_shares_df))

failed_shares_df

In [ ]:
def add_market_cap_from_shares(processed, shares_outstanding):
    """
    Aggiunge shares_outstanding e market_cap al dataset.
    Usa merge_asof per portare in ogni data l'ultimo numero di azioni disponibile.
    """
    ## copio e riordino i due dataframe 

    df = processed.copy()
    df = df.sort_values(["Symbol", "Date"])
    
    if shares_outstanding.empty:
        df["shares_outstanding"] = np.nan
        df["market_cap"] = np.nan
        return df
    
    shares = shares_outstanding.copy()
    shares = shares.sort_values(["Symbol", "Date"])
    
    result = []
    
    for symbol in df["Symbol"].unique(): # per ogni azienda
        df_s = df[df["Symbol"] == symbol].copy() # seleziono il sotto-dataframe relativo all'azienda con prezzi, rendimenti e denaro scambiato
        sh_s = shares[shares["Symbol"] == symbol].copy() # seleziono il sotto-dataframe per le azioni in circolazione
        
        if sh_s.empty:
            df_s["shares_outstanding"] = np.nan
            df_s["market_cap"] = np.nan
        else:
            df_s = pd.merge_asof(
                df_s.sort_values("Date"),
                sh_s[["Date", "shares_outstanding"]].sort_values("Date"),
                on="Date",
                direction="backward"
            ) # uniamo i due dataframe, in ordine cronologico, scegliendo per ogni data l'ultimo valore di shares_oustanding disponibile (backwards)
            
            df_s["market_cap"] = df_s["P_raw"] * df_s["shares_outstanding"] # calcolo della capitalizzazione di mercato
        
        result.append(df_s)
    
    return pd.concat(result, ignore_index=True)

In [ ]:
# aggiungiamo quindi la capitalizzazione di mercato

processed_full = add_market_cap_from_shares(
    processed=processed,
    shares_outstanding=shares_outstanding
)

processed_full_path = os.path.join(
    PROCESSED_DIR,
    "italy_full_dataset_2015_2024.csv"
)

processed_full.to_csv(processed_full_path, index=False)

processed_full.head()

In [ ]:
# costruzione matrice di capitalizzazione di mercato

market_cap_matrix = (
    processed_full[processed_full["Symbol"].isin(eligible_symbols)] # per titoli validi
    .pivot(index="Date", columns="Symbol", values="market_cap")
    .sort_index()
)

market_cap_matrix_path = os.path.join(
    PROCESSED_DIR,
    "market_cap_matrix_italy_2015_2024.csv"
)

market_cap_matrix.to_csv(market_cap_matrix_path)

market_cap_availability = market_cap_matrix.notna().mean().sort_values()

display(market_cap_matrix.head())
display(market_cap_availability)

In [ ]:
returns_final = returns_matrix.copy()

# imposto soglia massima di valori di ritorni mancanti
max_missing_per_day = int(0.10 * returns_final.shape[1])

returns_final = returns_final[
    returns_final.isna().sum(axis=1) <= max_missing_per_day # applichiamo il filtro
]

returns_final_path = os.path.join(
    PROCESSED_DIR,
    "returns_final_for_mst_italy_2015_2024.csv"
)

returns_final.to_csv(returns_final_path)

print("Dimensioni matrice rendimenti:", returns_final.shape)

returns_final.head()

In [ ]:
# aziende effettivamente utilizzate

company_info = (
    tickers[tickers["symbol"].isin(eligible_symbols)]
    .copy()
    .sort_values("symbol")
)

company_info_path = os.path.join(
    PROCESSED_DIR,
    "company_info_italy_2015_2024.csv"
)

company_info.to_csv(company_info_path, index=False)

company_info